<a href="https://colab.research.google.com/github/mzaib1012/CAN-Bus-Network-Simulator/blob/main/python_simulator/CAN_Bus_Arbitration_Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Block 1: Installation and Library Imports
!pip install python-can

import time
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import can

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.2.0
    Uninstalling wrapt-2.2.0:
      Successfully uninstalled wrapt-2.2.0


In [2]:
# Block 2: ECU Node Configurations
class ECU:
    def __init__(self, name, can_id, data_payload):
        self.name = name
        self.can_id = can_id  # Integer ID
        # Convert integer ID to an 11-bit binary representation string, then to a list of ints
        self.binary_id = [int(bit) for bit in f"{can_id:011b}"]
        self.data_payload = data_payload

# Initialize our vehicle nodes
powertrain = ECU(name="Powertrain (PCM)", can_id=0x100, data_payload=[0x22, 0x1A]) # High Priority
bms = ECU(name="Battery Mgmt (BMS)", can_id=0x200, data_payload=[0x64, 0xFF])     # Medium Priority
dashboard = ECU(name="Dashboard Cluster", can_id=0x300, data_payload=[0x05, 0x12]) # Low Priority

nodes = [powertrain, bms, dashboard]

# Quick print to verify binary IDs
for node in nodes:
    print(f"{node.name} -> ID: {hex(node.can_id)} -> Binary ID bits: {node.binary_id}")

Powertrain (PCM) -> ID: 0x100 -> Binary ID bits: [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
Battery Mgmt (BMS) -> ID: 0x200 -> Binary ID bits: [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Dashboard Cluster -> ID: 0x300 -> Binary ID bits: [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]


In [3]:
# Block 3: Simulating Bit-by-Bit Bus Arbitration
def simulate_bus_arbitration(active_nodes):
    print("--- Starting CAN Bus Arbitration Cycle ---")
    bus_line = []

    # Track which nodes are still actively competing at each clock bit
    competing_nodes = list(active_nodes)
    eliminated_nodes = []

    # CAN standard identifier has 11 bits
    for bit_idx in range(11):
        print(f"\n[Bit Clock {bit_idx+1}/11]")

        # Gather the bit each remaining ECU wants to send
        bits_to_send = {node.name: node.binary_id[bit_idx] for node in competing_nodes}
        for name, bit in bits_to_send.items():
            print(f"  {name} wants to send: {bit} ({'Recessive' if bit==1 else 'Dominant'})")

        # Physical wired-AND/Wired-OR behavior: If any bit is 0 (Dominant), the bus becomes 0.
        # Otherwise, if all are 1, the bus stays 1.
        actual_bus_state = 0 if 0 in bits_to_send.values() else 1
        bus_line.append(actual_bus_state)
        print(f"  >> RESULTING BUS STATE: {actual_bus_state}")

        # Check for losers: if a node sent a 1 but the bus read a 0, it loses arbitration
        still_in = []
        for node in competing_nodes:
            if bits_to_send[node.name] == 1 and actual_bus_state == 0:
                print(f"  ❌ {node.name} lost arbitration and dropped off the bus.")
                eliminated_nodes.append((node.name, bit_idx + 1))
            else:
                still_in.append(node)

        competing_nodes = still_in

        # If only one winner remains, they control the bus and can finish uninterrupted
        if len(competing_nodes) == 1:
            winner = competing_nodes[0]
            print(f"\n🏆 Winner Decided at bit {bit_idx+1}: {winner.name}!")
            # The remaining bits of the winner's ID naturally populate the rest of the arbitration field
            for remaining_bit in winner.binary_id[bit_idx+1:]:
                bus_line.append(remaining_bit)
            break

    return winner, bus_line, eliminated_nodes

# Run the simulation
winner_node, final_bus_bits, logs = simulate_bus_arbitration(nodes)

--- Starting CAN Bus Arbitration Cycle ---

[Bit Clock 1/11]
  Powertrain (PCM) wants to send: 0 (Dominant)
  Battery Mgmt (BMS) wants to send: 0 (Dominant)
  Dashboard Cluster wants to send: 0 (Dominant)
  >> RESULTING BUS STATE: 0

[Bit Clock 2/11]
  Powertrain (PCM) wants to send: 0 (Dominant)
  Battery Mgmt (BMS) wants to send: 1 (Recessive)
  Dashboard Cluster wants to send: 1 (Recessive)
  >> RESULTING BUS STATE: 0
  ❌ Battery Mgmt (BMS) lost arbitration and dropped off the bus.
  ❌ Dashboard Cluster lost arbitration and dropped off the bus.

🏆 Winner Decided at bit 2: Powertrain (PCM)!


In [4]:
# Block 4: Output Log Generation
print("\n" + "="*40)
print("FINAL TRANSMISSION SUCCESS")
print("="*40)

# Create a python-can Frame Object for the winner
can_frame = can.Message(
    arbitration_id=winner_node.can_id,
    data=winner_node.data_payload,
    is_extended_id=False
)

print(f"Successfully Transmitted Frame:")
print(f"Timestamp: {time.time()}")
print(f"Channel: Virtual_Vehicle_Bus_0")
print(f"Identifier (Hex): {hex(can_frame.arbitration_id)}")
print(f"DLC: {can_frame.dlc}")
print(f"Data Payload: [{', '.join([hex(b) for b in can_frame.data])}]")


FINAL TRANSMISSION SUCCESS
Successfully Transmitted Frame:
Timestamp: 1779869131.5672898
Channel: Virtual_Vehicle_Bus_0
Identifier (Hex): 0x100
DLC: 2
Data Payload: [0x22, 0x1a]
